# Exomoon detectability pipeline v31 - FATSO beta

Main grid pipeline with optional nested-sampling validation.

* Default target: **Kepler-1708 b**.

* Nested sampling results are fundamentally separated from the grid run results themselves and each function without the other.

* Nested validation: **best_case** for Bayes-factor validation, **worst_case** for weak/non-detection behaviour, and **intermediate_case** for posterior-shape/corner-plot.

* For other targets, copy a config block from **exomoon_detectability_v24_extra_target_blocks.py** into the first cell.


## White-dwarf object issues: WD1856 b and Helix Nebula b
Not a typical small-planet transit geometry - planet are enormous relative to hosts. 
Pandora is not fit to do this, therefore the code will default to the **fast** option. The **fast** option skips over photodynamical constraints and so e.g. WD1856 are dynamical/geometry stress tests, not clean exomoon validation cases.

**WD1856 b:** The planet is enormous relative to the white dwarf, so the event is closer to a compact eclipse than a standard planet crossing a large stellar disk. 

**Helix Nebula b:** Even less clean as a transit experiment. The proposed companion interpretation is based on periodic variability, not a confirmed transiting planet, and the companion radius can be comparable to the central white dwarf. i.e., Even the planet is still tentative.


In [1]:
import time
import math
import matplotlib.pyplot as plt
import numpy as np
import importlib

import exomoon_detectability_exomoon_v31_simple_manual_transits as exo

exo = importlib.reload(exo)
print("Loaded:", exo.__file__)

# Default target: Kepler-1708 b.
# Replace the configuration block below to run a different target.

# Editable detection thresholds.
# A trigger requires BOTH SNR >= SNR_THRESHOLD and logK_proxy >= LOGK_THRESHOLD.
SNR_THRESHOLD = 7.0
LOGK_THRESHOLD = math.log(10.0)

cfg = exo.Config(
    target="kepler1708",
    star=exo.Star(
        name="Kepler-1708",
        mass_msun=1.061,
        radius_rsun=1.141,
        limb_darkening_u1=0.43,
        limb_darkening_u2=0.26,
    ),
    planet=exo.Planet(
        name="Kepler-1708 b",
        period_days=737.11,
        radius_rj=0.89,
        mass_mj=1.0,
        semi_major_au=1.64,
        impact_b=0.4,
    ),
    moon_grid=exo.MoonGrid(
        radius_re_min=2.0,
        radius_re_max=3.2,
        a_rp_min=8.0,
        hill_fraction_max=0.08,
        mutual_inc_deg_max=2.0,
        ecc_min=0.0,
        ecc_max=0.10,
    ),
    noise=exo.telescope_noise_preset(
        instrument="kepler",
        target_mag=16.3,
        red_ppm=120.0,
        red_timescale_hr=8.0,
        duty_cycle=0.87,
    ),
    n_transits=2,
    local_window_days=12.4,
    rng_seed=1,
    snr_threshold=SNR_THRESHOLD,
    logk_threshold=LOGK_THRESHOLD,
)

# Keep as "auto" for the intended rule: Pandora truth for Kepler targets, fast truth for WD targets.
# If Pandora/pandoramoon is unavailable, or if you want the fully self-consistent fast test, set this to "fast".
truth_backend = "auto"


Loaded: C:\Users\janse\PSLU Python code\P3\P3 detectability\FATSO_beta\Finale2\exomoon_detectability_exomoon_v31_simple_manual_transits.py


In [2]:
# Display parameter inputs.
for name, table in exo.parameter_input_dataframes(cfg).items():
    print("\n" + name)
    display(table)


star


,name,mass_msun,radius_rsun,limb_darkening_u1,limb_darkening_u2
0,Kepler-1708,1.061,1.141,0.43,0.26



planet


,name,period_days,radius_rj,mass_mj,semi_major_au,t0_days,impact_b,ecc,omega_deg
0,Kepler-1708 b,737.11,0.89,1.0,1.64,0.0,0.4,0.0,90.0



moon_grid


,radius_re_min,radius_re_max,a_rp_min,hill_fraction_max,mutual_inc_deg_max,ecc_min,ecc_max,force_coplanar
0,2.0,3.2,8.0,0.08,2.0,0.0,0.1,False



noise


,instrument,cadence_min,white_ppm,red_ppm,red_timescale_hr,duty_cycle
0,kepler,29.4,527.701871,120.0,8.0,0.87



experiment


,target,n_transits,local_window_days,rng_seed,snr_threshold,logk_threshold
0,kepler1708,2,12.4,1,7.0,2.302585


## Run-size presets

The grid run size controls how densely the moon-parameter and no-moon noise spaces are sampled.

| Preset     | Moon-radius grid points | Moon-separation grid points | Red-noise amplitude grid points | Red-noise timescale grid points | Repeats per grid cell | Injected-moon tests | No-moon noise tests | Total grid tests | Intended use                       |
| ---------- | ----------------------: | --------------------------: | ------------------------------: | ------------------------------: | --------------------: | ------------------: | ------------------: | ---------------: | ---------------------------------- |
| `smoke`    |                       3 |                           3 |                               3 |                               3 |                     2 |                  18 |                  18 |               36 | Check runs.      |
| `quick`    |                       6 |                           6 |                               5 |                               5 |                     3 |                 108 |                  75 |              183 | Fast development run.              |
| `standard` |                      10 |                          10 |                               8 |                               8 |                    10 |                1000 |                 640 |             1640 | Main run.             |
| `long`     |                      30 |                          30 |                              12 |                              12 |                    20 |               18000 |                2880 |            20880 | Higher-resolution final run. |

The test counts are:

$ N_\mathrm{moon} = N_R \times N_a \times N_\mathrm{per,cell}$

$ N_\mathrm{noise} = N_\mathrm{red} \times N_\tau \times N_\mathrm{per,cell}$

For notebook checks, use `smoke` or `quick`.
For final reported plots, use `standard` at minimum.
Use `long` only when the setup is already working and the extra runtime is justified.


In [3]:
t0 = time.time()
results = exo.run_grid_suite(
    cfg,
    n_radius=30,       
    n_separation=30,  
    n_red=12,          
    n_tau=12,           
    n_per_cell=20,
    truth_backend=truth_backend,
    snr_threshold=SNR_THRESHOLD,
    logk_threshold=LOGK_THRESHOLD,
    progress=True,
)
print(f"Grid runtime: {time.time() - t0:.2f} seconds")
print(f"Truth backend used: {results['truth_backend']}")
print(f"Recovery planet baseline used: {results['recovery_planet_backend']}")
print(f"SNR threshold: {results['cfg'].snr_threshold}")
print(f"logK_proxy threshold: {results['cfg'].logk_threshold:.3f}")


moon grid 1/900: Rm=2 Re, a=8 Rp
moon grid 2/900: Rm=2 Re, a=8.44 Rp
moon grid 3/900: Rm=2 Re, a=8.87 Rp
moon grid 4/900: Rm=2 Re, a=9.31 Rp
moon grid 5/900: Rm=2 Re, a=9.74 Rp
moon grid 6/900: Rm=2 Re, a=10.2 Rp
moon grid 7/900: Rm=2 Re, a=10.6 Rp
moon grid 8/900: Rm=2 Re, a=11.1 Rp
moon grid 9/900: Rm=2 Re, a=11.5 Rp
moon grid 10/900: Rm=2 Re, a=11.9 Rp
moon grid 11/900: Rm=2 Re, a=12.4 Rp
moon grid 12/900: Rm=2 Re, a=12.8 Rp
moon grid 13/900: Rm=2 Re, a=13.2 Rp
moon grid 14/900: Rm=2 Re, a=13.7 Rp
moon grid 15/900: Rm=2 Re, a=14.1 Rp
moon grid 16/900: Rm=2 Re, a=14.5 Rp
moon grid 17/900: Rm=2 Re, a=15 Rp
moon grid 18/900: Rm=2 Re, a=15.4 Rp
moon grid 19/900: Rm=2 Re, a=15.9 Rp
moon grid 20/900: Rm=2 Re, a=16.3 Rp
moon grid 21/900: Rm=2 Re, a=16.7 Rp
moon grid 22/900: Rm=2 Re, a=17.2 Rp
moon grid 23/900: Rm=2 Re, a=17.6 Rp
moon grid 24/900: Rm=2 Re, a=18 Rp
moon grid 25/900: Rm=2 Re, a=18.5 Rp
moon grid 26/900: Rm=2 Re, a=18.9 Rp
moon grid 27/900: Rm=2 Re, a=19.3 Rp
moon grid 28/900:

KeyboardInterrupt: 

In [ ]:
# Plot recovery probability heatmaps and rednoise FPP
fig, ax = exo.plot_recovery_heatmap(results)
plt.show()

fig, ax = exo.plot_uncertainty_heatmap(results)
plt.show()

fig, ax = exo.plot_false_positive_heatmap(results)
plt.show()


In [ ]:
print(results["report"])

The mean recovery and per-cell mean recovery are
identical since each grid cell receives the same number of
injection trials; they would differ in the presence of missing runs,
rejected cases, NaNs, or uneven grid sampling, so agreement here confirms
the pipeline ran uniformly across the full radius--separation grid.

In [ ]:
moon = results["moon_trials"]
noise = results["noise_trials"]

k_rec = int(moon["detected"].sum())
n_moon = len(moon)

k_fp = int(noise["detected"].sum())
n_noise = len(noise)

TPR = k_rec / n_moon if n_moon else np.nan
FPR = k_fp / n_noise if n_noise else np.nan

K_grid_proxy = TPR / FPR if FPR > 0 else np.inf
lnK_grid_proxy = np.log(K_grid_proxy) if np.isfinite(K_grid_proxy) else np.inf

print("Whole-grid trigger summary")
print("==========================")
print(f"Injected moons recovered: {k_rec} / {n_moon} = {TPR:.4f} ({100*TPR:.2f}%)")
print(f"No-moon false positives: {k_fp} / {n_noise} = {FPR:.4f} ({100*FPR:.2f}%)")
print(f"K_grid_proxy = TPR/FPR = {K_grid_proxy:.6g}")
print(f"ln(K_grid_proxy) = {lnK_grid_proxy:.6g}")
print()

print("Per-trial logK_proxy distribution")
print("=================================")
print("Injected moon trials:")
display(moon["logk_proxy"].describe())

print("No-moon trials:")
display(noise["logk_proxy"].describe())

In [ ]:
paths = exo.save_grid_results(results, output_dir="results")
for key, path in paths.items():
    print(f"{key}: {path}")


In [ ]:
# Optional nested-sampling Bayesian validation.
# best_case: Bayes-factor stress test
# worst_case: weak/non-detection diagnostic
# intermediate_case: posterior-shape diagnostic
RUN_NESTED = False
nested = None

if RUN_NESTED:
    ns_cfg = exo.NestedConfig(
        min_live_points=200,
        dlogz=0.5,
        max_ncalls=None,
        use_stepsampler=True,
        stepsampler_nsteps=50,
        stepsampler_adaptive_nsteps="move-distance",
        stepsampler_region_filter=True,
        frac_remain=0.5,
    )

    t0 = time.time()
    nested = exo.run_nested_validation(
        cfg,
        ns_cfg=ns_cfg,
        truth_backend=truth_backend,
        output_dir="results_v31_nested",
    )
    print(f"Nested validation runtime: {time.time() - t0:.2f} seconds")
    print(nested["report"])
    display(nested["comparison"])

    for case_name in nested["cases"]:
        fig, ax = exo.plot_nested_validation(nested, case=case_name)
        plt.show()

In [ ]:
# Plot the intermediate-case posterior after running nested validation.
if nested is not None:
    fig = exo.plot_nested_corner(nested, case="intermediate_case")
    plt.show()
else:
    print("Run the nested-validation cell before creating the corner plot.")

#### v31 treatment notes

**Main grid / FPP treatment**

v31 restores the v27a fast-backend detection trigger:

$$\Delta\chi^2 = \chi^2_0 - \chi^2_1,$$

with the matched-filter SNR computed from the fitted box depth and the per-point uncertainty.

**Nested validation**

The nested validation still uses three deterministic injected-moon cases with separated diagnostic roles:

- **best_case**: Bayes-factor validation only. The posterior may be too concentrated for a meaningful corner plot.
- **worst_case**: weak/non-detection behaviour. Broad or weakly constrained posteriors are expected.
- **intermediate_case**: posterior-shape and corner-plot diagnostics.

For each case the notebook reports the Bayesian model comparison.

where $M0$ is the planet-only model and $M1$ is the planet+moon model.


### Bayes-factor interpretation using Jeffreys' scale

For the nested-sampling comparison,

$$
\ln K = \ln Z_{M1} - \ln Z_{M0},
$$

| $K = Z_{M1}/Z_{M0}$ | $\log_{10}K$ | $\ln K$ | Interpretation for $M1$ over $M0$ |
|---:|---:|---:|---|
| $1 \text{ to } 10^{1/2}$ | 0 to 0.5 | 0 to 1.15 | Barely worth mentioning |
| $10^{1/2} \text{ to } 10$ | 0.5 to 1.0 | 1.15 to 2.30 | Substantial |
| $10 \text{ to } 10^{3/2}$ | 1.0 to 1.5 | 2.30 to 3.45 | Strong |
| $10^{3/2} \text{ to } 10^2$ | 1.5 to 2.0 | 3.45 to 4.61 | Very strong |
| $>10^2$ | \(>2.0\) | \(>4.61\) | Decisive |

These labels are interpretive guidelines, not detection thresholds. In this simplified version, detection/false-positive statements come from the main injection/recovery grid and no-moon/no-object grid, while the nested run uses best/worst/intermediate diagnostic roles.